# NYC Motor Vehicle Collisions — Exploratory Data Analysis

**Dataset:** [Motor Vehicle Collisions – Crashes (NYC Open Data)](https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95/about_data)  
**Time span:** January 2023 – December 2025  
**Records:** 270,000+  

### Notebook Structure
1. [Imports & Data Loading](#1-imports--data-loading)
2. [Initial Data Inspection](#2-initial-data-inspection)
3. [Data Cleaning](#3-data-cleaning)
   - 3a. Column Selection
   - 3b. Missing Value Imputation (Borough & ZIP Code via Spatial Join)
   - 3c. Feature Engineering (Datetime Columns)
4. [Interactive Map Data Prep](#4-interactive-map-data-prep)
5. [Exploratory Data Analysis (EDA)](#5-exploratory-data-analysis)
   - 5a. Time-Series Analysis
   - 5b. Spatial / Borough Analysis
   - 5c. Casualty Analysis

---
## 1. Imports & Data Loading

In [ ]:
# --- Core libraries ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Geospatial libraries ---
import geopandas as gpd          # GeoDataFrames + spatial joins
import shapely as wkt            # Geometry parsing helpers
import folium                    # Interactive map rendering

# --- Utilities ---
from pathlib import Path         # OS-agnostic file paths

In [ ]:
# Set the working directory to the folder containing this notebook,
# then load the three source files.
base = Path.cwd()

df = pd.read_csv(base / "Crashes_20251231.csv")   # Main crash records
bb = pd.read_csv(base / "BoroughBoundaries.csv")  # Borough boundary polygons (WKT)
zc = pd.read_csv(base / "ModifiedZipCode.csv")    # ZIP code boundary polygons (WKT)

---
## 2. Initial Data Inspection

In [ ]:
# Quick look at the first five rows
df.head(5)

In [ ]:
# Dataset dimensions and duplicate check
n_rows, n_cols = df.shape
print("Number of rows:", n_rows)
print("Number of columns:", n_cols)
print("=" * 60)
print("Duplicated rows:", df.duplicated().sum())

---
## 3. Data Cleaning

### 3a. Column Selection

In [ ]:
# Keep only the columns needed for analysis:
#   cols 0-6  → crash date/time, location identifiers (borough, zip, lat, lon)
#   cols 10-11 → injury / death counts
#   col  23   → collision ID (used as a unique crash key)
df = df.iloc[:, np.r_[0:7, 10:12, 23]].copy()
df.info()

In [ ]:
# Identify columns that still have missing values and their % missing
missing_percent = df.isnull().mean() * 100
print(missing_percent[missing_percent > 0])

### 3b. Missing Value Imputation via Spatial Join

Many rows are missing `BOROUGH` or `ZIP CODE` but have valid latitude/longitude.
We use a **spatial join** — matching crash coordinates to polygon boundaries — to fill those gaps.

In [ ]:
# Build a GeoDataFrame of borough polygons from the WKT geometry column.
# EPSG:4326 is the standard coordinate reference system for GPS lat/lon.
bb["geometry"] = gpd.GeoSeries.from_wkt(bb["the_geom"])
gdf_borough = gpd.GeoDataFrame(bb, geometry="geometry", crs="EPSG:4326")
gdf_borough.head()

In [ ]:
# Impute missing BOROUGH values using spatial join with borough polygons.
# Strategy: select rows where BOROUGH is null but coordinates are available,
# convert to points, then join to find which borough polygon each point falls in.

mask_borough_fill = (
    df["BOROUGH"].isna()
    & df["LATITUDE"].notna()
    & df["LONGITUDE"].notna()
    & (df["LATITUDE"] != 0)
    & (df["LONGITUDE"] != 0)
)

df_borough_fill = df.loc[mask_borough_fill].copy()

# Convert crash rows to a GeoDataFrame of point geometries
gdf_crash_bb = gpd.GeoDataFrame(
    df_borough_fill,
    geometry=gpd.points_from_xy(df_borough_fill["LONGITUDE"], df_borough_fill["LATITUDE"]),
    crs="EPSG:4326"
)

# Spatial join: each crash point → the borough polygon it lies within
gdf_joined_bb = gpd.sjoin(
    gdf_crash_bb,
    gdf_borough[["BoroName", "geometry"]],
    how="left",
    predicate="within"
)

borough_joined = gdf_joined_bb["BoroName"].str.upper()

# Write the imputed borough names back into the main dataframe
df.loc[borough_joined.index, "BOROUGH"] = borough_joined
print(f"% missing BOROUGH after imputation: {df['BOROUGH'].isna().mean() * 100:.2f}%")

In [ ]:
# Build a GeoDataFrame of modified ZIP code polygons (MODZCTA = Modified Zip Code Tabulation Area)
zc["geometry"] = gpd.GeoSeries.from_wkt(zc["the_geom"])
gdf_zipcode = gpd.GeoDataFrame(zc, geometry="geometry", crs="EPSG:4326")
gdf_zipcode.head()

In [ ]:
# Impute missing ZIP CODE values using the same spatial join approach.
mask_zipcode_fill = (
    df["ZIP CODE"].isna()
    & df["LATITUDE"].notna()
    & df["LONGITUDE"].notna()
    & (df["LATITUDE"] != 0)
    & (df["LONGITUDE"] != 0)
)
df_zipcode_fill = df.loc[mask_zipcode_fill].copy()

gdf_crash_zc = gpd.GeoDataFrame(
    df_zipcode_fill,
    geometry=gpd.points_from_xy(df_zipcode_fill["LONGITUDE"], df_zipcode_fill["LATITUDE"]),
    crs="EPSG:4326"
)

gdf_joined_zc = gpd.sjoin(
    gdf_crash_zc,
    gdf_zipcode[["MODZCTA", "geometry"]],
    how="left",
    predicate="within"
)

# Align types: convert both to nullable Int64 then string so they merge cleanly
zipcode_joined = gdf_joined_zc["MODZCTA"].astype("Int64").astype("string")
df["ZIP CODE"] = df["ZIP CODE"].astype("Int64").astype("string")
df.loc[zipcode_joined.index, "ZIP CODE"] = zipcode_joined
print(f"% missing ZIP CODE after imputation: {df['ZIP CODE'].isna().mean() * 100:.2f}%")

In [ ]:
# Final missing value check after both imputations
missing_percent = df.isnull().mean() * 100
print(missing_percent[missing_percent > 0])

### 3c. Feature Engineering — Datetime Columns

In [ ]:
# Combine the separate date and time columns into a single datetime column
df["CRASH_DATETIME"] = pd.to_datetime(df["CRASH DATE"] + " " + df["CRASH TIME"])
print(f"Data ranges from {df['CRASH_DATETIME'].min()} to {df['CRASH_DATETIME'].max()}")

In [ ]:
# Extract granular time features for time-series analysis
df["YEAR"]       = df["CRASH_DATETIME"].dt.to_period("Y").astype(str)  # e.g. "2023"
df["YEAR_MONTH"] = df["CRASH_DATETIME"].dt.to_period("M").astype(str)  # e.g. "2023-01"
df["MONTH"]      = df["CRASH_DATETIME"].dt.month_name()                 # e.g. "January"
df["HOUR"]       = df["CRASH_DATETIME"].dt.hour                         # 0–23
df["WEEKDAY"]    = df["CRASH_DATETIME"].dt.day_name()                   # e.g. "Monday"

In [ ]:
# Quick sanity check after all cleaning and feature engineering
df.info()
df.head()

---
## 4. Interactive Map Data Prep

Aggregate crash counts, injuries, and deaths **per ZIP code per month**,
then join to ZIP code polygon geometries.  
The resulting GeoJSON file is consumed by the `Map.py` Streamlit app.

In [ ]:
# Aggregate: monthly crash count, injuries, and deaths by (Year, Month, Borough, ZIP)
monthly_crashes = (
    df.groupby(["YEAR", "MONTH", "BOROUGH", "ZIP CODE"])
    .agg(
        **{"Monthly crashes": ("COLLISION_ID", "count"),
           "Monthly injuries": ("NUMBER OF PERSONS INJURED", "sum"),
           "Monthly deaths": ("NUMBER OF PERSONS KILLED", "sum")}
    )
    .reset_index()
)

monthly_crashes.head()

In [ ]:
# Join monthly aggregates to ZIP code polygon geometries so each row has a shape for mapping.
# Drop rows where no matching geometry was found (ZIP codes outside NYC boundaries).
gdf_zipcode["MODZCTA"] = gdf_zipcode["MODZCTA"].astype("string")

monthly_crashes_joined = monthly_crashes.merge(
    gdf_zipcode[["MODZCTA", "geometry"]],
    how="left",
    left_on="ZIP CODE",
    right_on="MODZCTA"
).drop(columns="MODZCTA")

monthly_crashes_gdf = gpd.GeoDataFrame(monthly_crashes_joined, geometry="geometry", crs="EPSG:4326")
monthly_crashes_gdf = monthly_crashes_gdf.dropna(subset=["geometry"]).copy()

monthly_crashes_gdf.info()

# Export to GeoJSON — this file is read by Map.py at runtime
monthly_crashes_gdf.to_file("monthly_crashes_gdf.geojson")

---
## 5. Exploratory Data Analysis

### Helper Function

In [ ]:
def barplot_crash_counts(df, column, ax=None):
    """
    Plot a bar chart of crash counts for a given categorical column.

    Parameters
    ----------
    df     : pd.DataFrame  — source dataframe
    column : str           — column whose value counts to plot
    ax     : Axes, optional — existing Axes to draw on; creates one if None
    """
    counts = df[column].value_counts().sort_index()

    if ax is None:
        ax = plt.gca()

    counts.plot(kind="bar", ax=ax, label=f"Crash counts by {column}")
    ax.set_title(f"Crash Counts by {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Number of crashes")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(loc="upper right", bbox_to_anchor=(1, 1.15))
    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle="--", color="grey", alpha=0.5)

### 5a. Time-Series Analysis

**Key findings:**
- Annual crash counts declined steadily from 2023 through 2025.
- Monthly volumes fluctuated between ~6,000–8,000; winter months (Jan–Feb) had the fewest crashes.
- Fridays had the highest crash volumes; weekends were generally safer.
- Peak daily hour: 3 PM–5 PM; a notable midnight spike also appears at hour 0.

In [ ]:
# Annual, Year-Month, and calendar-Month distributions
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 12))
barplot_crash_counts(df, "YEAR",       ax=axes[0])
barplot_crash_counts(df, "YEAR_MONTH", ax=axes[1])
barplot_crash_counts(df, "MONTH",      ax=axes[2])
plt.tight_layout()
plt.show()

In [ ]:
# Enforce natural ordering for weekday categories
week_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["WEEKDAY"] = pd.Categorical(df["WEEKDAY"], categories=week_order, ordered=True)

# Day-of-week and hour-of-day distributions
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 8))
barplot_crash_counts(df, "WEEKDAY", ax=axes[0])
barplot_crash_counts(df, "HOUR",    ax=axes[1])
plt.tight_layout()
plt.show()

### 5b. Spatial / Borough Analysis

**Key findings:**
- Brooklyn and Queens had the highest total crash volumes (70,000+ each).
- Brooklyn and the Bronx trended consistently downward 2023→2025.
- Queens and Manhattan initially declined but rebounded in 2025.
- Staten Island was the lowest-volume borough throughout.

In [ ]:
# Note: cell referencing 'BOROUGH_FILLED' below is kept for reference;
# in the cleaned dataset the column is called 'BOROUGH'.
df["BOROUGH"].unique()

In [ ]:
# Enforce a logical display order for the five boroughs
borough_order = ["BROOKLYN", "QUEENS", "MANHATTAN", "BRONX", "STATEN ISLAND"]
df["BOROUGH"] = pd.Categorical(df["BOROUGH"], categories=borough_order, ordered=True)

# Total crash count per borough (all years combined)
fig, ax = plt.subplots(figsize=(12, 3))
barplot_crash_counts(df, "BOROUGH")
plt.show()

In [ ]:
# Grouped bar chart: crash counts by borough, broken down by year.
# unstack() pivots the 'YEAR' index level into separate columns for side-by-side bars.
stratified_counts = (
    df.groupby(["BOROUGH", "YEAR"], observed=True)
    .size()
    .unstack()
    .sort_values(by="2025", ascending=False)
)

stratified_counts.plot(kind="bar", figsize=(10, 4))
plt.title("Crash Counts by Borough and Year")
plt.xlabel("Borough")
plt.ylabel("Number of crashes")
plt.xticks(rotation=0)
ax = plt.gca()
ax.set_axisbelow(True)
ax.grid(axis="y", linestyle="--", color="grey", alpha=0.5)
plt.legend(title="Year")
plt.tight_layout()
plt.show()

In [ ]:
# Line chart: monthly crash trends per borough (2023–2025).
# Using .T to transpose so months are on the x-axis and boroughs are the series.
stratified_counts_ym = (
    df.groupby(["BOROUGH", "YEAR_MONTH"], observed=True)
    .size()
    .unstack()
)

stratified_counts_ym.T.plot(kind="line", figsize=(12, 4), marker="o")
plt.title("Crash Trends from 2023 to 2025 by Borough")
plt.xlabel("Year-Month")
plt.ylabel("Number of crashes")
plt.grid(True, linestyle="--", alpha=0.7)
plt.legend(title="Borough", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: crash frequency by borough (rows) and hour of day (columns).
# Useful to spot borough-specific peak hours at a glance.
pivot_table = df.pivot_table(
    index="BOROUGH",
    columns="HOUR",
    values="COLLISION_ID",
    aggfunc="count",
    observed=True
)
plt.figure(figsize=(16, 3))
sns.heatmap(pivot_table, cmap="YlOrRd", square=True)
plt.title("Crash Heatmap by Borough and Hour")
plt.tight_layout()
plt.show()

### 5c. Casualty Analysis

**Key findings:**
- 43% of crashes resulted in injuries; only 0.27% were fatal.
- Brooklyn and Queens led in raw casualty counts but had lower *fatality rates*.
- Staten Island had the highest fatality rate (deaths / total casualties).
- 2025 was the safest year, with fatality rates converging around 0.40% across all boroughs.

In [ ]:
# ZIP code-level totals: used for the distribution histograms below
total_crashes = (
    df.groupby("ZIP CODE")
    .agg(
        **{"Total crashes":  ("COLLISION_ID", "count"),
           "Total injuries": ("NUMBER OF PERSONS INJURED", "sum"),
           "Total deaths":   ("NUMBER OF PERSONS KILLED", "sum")}
    )
    .reset_index()
)
total_crashes.head()

In [ ]:
# Distribution of total crashes, injuries, and deaths across ZIP codes.
# Right-skewed distributions are expected: most ZIPs have moderate counts,
# a few high-density areas see much higher volumes.
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].hist(total_crashes["Total crashes"],  bins=20, color="skyblue",  edgecolor="white")
axes[0].set_title("Distribution of Total Crashes")
axes[0].set_xlabel("Crashes per ZIP Code")
axes[0].set_ylabel("Number of ZIP Codes")

axes[1].hist(total_crashes["Total injuries"], bins=20, color="c",        edgecolor="white")
axes[1].set_title("Distribution of Total Injuries")
axes[1].set_xlabel("Injuries per ZIP Code")

axes[2].hist(total_crashes["Total deaths"],   bins=15, color="orangered", edgecolor="white")
axes[2].set_title("Distribution of Total Deaths")
axes[2].set_xlabel("Fatalities per ZIP Code")

plt.tight_layout()
plt.show()

In [ ]:
# Summarise overall crash severity: fatal vs. injury vs. property-damage-only
death_count   = (df["NUMBER OF PERSONS KILLED"] != 0).sum()
injury_count  = ((df["NUMBER OF PERSONS INJURED"] != 0) & (df["NUMBER OF PERSONS KILLED"] == 0)).sum()
crash_count   = len(df)

injured_rate    = injury_count  / crash_count
fatal_rate      = death_count   / crash_count
no_injury_rate  = 1.0 - (injured_rate + fatal_rate)

labels  = ["Injury Crashes", "Fatal Crashes", "Property Damage Only\n(No Injury)"]
sizes   = [injured_rate, fatal_rate, no_injury_rate]
colors  = ["c", "orangered", "skyblue"]
explode = (0.1, 0.2, 0)

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(sizes, explode=explode, labels=labels, colors=colors, autopct="%1.2f%%", startangle=140)
plt.title("Distribution of Crash Severities", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Borough-level injury vs. death counts on a log scale.
# Log scale is necessary because injuries vastly outnumber deaths,
# making it difficult to compare them on a linear axis.
borough_casualties = (
    df.groupby("BOROUGH", observed=True)[["NUMBER OF PERSONS INJURED", "NUMBER OF PERSONS KILLED"]]
    .sum()
    .sort_values("NUMBER OF PERSONS INJURED", ascending=False)
)

borough_casualties.plot(kind="bar", figsize=(10, 4), logy=True, color=["c", "orangered"])
plt.title("Casualties by Borough (Log Scale)", fontsize=14)
plt.xlabel("Borough", fontsize=12)
plt.ylabel("Count (log scale)", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", color="grey", alpha=0.5)
plt.legend(["Injured", "Killed"])
plt.tight_layout()
plt.show()

In [ ]:
# Fatality rate = deaths / (deaths + injuries) × 100
# Reveals *severity* independent of crash volume.
borough_casualties["Fatality Rate"] = (
    borough_casualties["NUMBER OF PERSONS KILLED"]
    / (borough_casualties["NUMBER OF PERSONS INJURED"] + borough_casualties["NUMBER OF PERSONS KILLED"])
) * 100

fig, ax = plt.subplots(figsize=(10, 4))
borough_casualties["Fatality Rate"].sort_values().plot(kind="bar", color="orangered", ax=ax)

ax.set_ylabel("Fatalities per 100 Casualties (%)")
ax.set_title("Crash Fatality Rate by Borough")
ax.set_xticklabels(borough_casualties.index, rotation=0)
ax.set_axisbelow(True)
ax.grid(axis="y", linestyle="--", color="grey", alpha=0.5)

# Annotate each bar with its exact percentage
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", (p.get_x() * 1.005, p.get_height() * 1.01))

plt.tight_layout()
plt.show()